In [1]:
print("OK")

OK


In [2]:
%pwd

'f:\\Hash Projects\\VitaAI-health-assistant\\research'

In [4]:
import os
os.chdir("F:\Hash Projects\VitaAI-health-assistant")

In [5]:
%pwd

'F:\\Hash Projects\\VitaAI-health-assistant'

In [6]:
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter


c:\Users\harsh\.conda\envs\vitaai\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
# Extract text from PDF files
def load_pdf_files(data):
    loader = DirectoryLoader(
        data,
        glob="*.pdf",
        loader_cls=PyPDFLoader
    )
     
    documents = loader.load()
    return documents

In [8]:
extracted_data = load_pdf_files("data")

PdfReadError("Invalid Elementary Object starting with b'P' @25528101: b'6 0 obj<</Universal PDF(The process that creates this PDF constitutes a trade se'")


In [10]:
extracted_data

[Document(metadata={'producer': 'CBS Publishers & Distributors Pvt., Ltd.', 'creator': 'www.eduport-global.com', 'creationdate': '', 'moddate': '2020-05-30T22:10:00+05:30', 'title': 'B. D. Chaurasia’s Human Anatomy: Regional & Applied Dissection & Clinical, Volume 3, Head and Neck and Volume 4, Brain–Neuroanatomy (Set of 2) - Krishna Garg, Pragati Sheel Mittal, Mrudula Chandrupatla - 8th Edition (2019) 640 pp., ISBN: 9789388902755', 'keywords': 'Human Anatomy', 'author': 'Krishna Garg, Pragati Sheel Mittal, Mrudula Chandrupatla', 'subject': 'Human Anatomy', 'source': 'data\\B D Chaurasia’s Human Anatomy Regional & Applied Dissection.pdf', 'total_pages': 640, 'page': 0, 'page_label': 'Cover'}, page_content=''),
 Document(metadata={'producer': 'CBS Publishers & Distributors Pvt., Ltd.', 'creator': 'www.eduport-global.com', 'creationdate': '', 'moddate': '2020-05-30T22:10:00+05:30', 'title': 'B. D. Chaurasia’s Human Anatomy: Regional & Applied Dissection & Clinical, Volume 3, Head and Nec

In [11]:
len(extracted_data)

10729

In [13]:
from typing import List
from langchain_core.documents import Document
"""
Given a list of Document objects, returns a new list of Document objects 
containing only 'source' in metadata and the original page_content.
"""
def filter_to_minimal_docs(docs: List[Document]) -> List[Document]:

    filter_data: List[Document] = []
    for doc in docs:
        src = doc.metadata.get("source")
        page_no = doc.metadata.get("page")
        filter_data.append(
            Document(
                page_content = doc.page_content,
                metadata={"source":src, "page":page_no}
            )
        )
    return filter_data

In [14]:
filter_data = filter_to_minimal_docs(extracted_data)

In [15]:
filter_data

[Document(metadata={'source': 'data\\B D Chaurasia’s Human Anatomy Regional & Applied Dissection.pdf', 'page': 0}, page_content=''),
 Document(metadata={'source': 'data\\B D Chaurasia’s Human Anatomy Regional & Applied Dissection.pdf', 'page': 1}, page_content='Regional and Applied Dissection and Clinical\nV olume3\nHuman\nAnatomy\nBD Chaurasia’s Eighth\nEdition\nHead and Neck\nAs per Medical Council of India: Competency based Undergraduate Curriculum for the Indian Medical Graduate , 2018'),
 Document(metadata={'source': 'data\\B D Chaurasia’s Human Anatomy Regional & Applied Dissection.pdf', 'page': 2}, page_content='Dr BD Chaurasia (1937–1985)\nwas Reader in Anatomy at GR Medical College, Gwalior.\nHe received his MBBS in 1960, MS in 1965 and PhD in 1975.\nHe was elected fellow of National Academy of Medical Sciences (India) in 1982.\nHe was a member of the Advisory Board of the Acta Anatomica since 1981,\nmember of the editorial board of Bionature, and in addition\nmember of a numb

In [16]:
len(filter_data)

10729

In [17]:
# Split  the documents into smaller chunks
def text_split(minimal_docs):
    text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=20,
    length_function=len
)
    texts_chunk = text_splitter.split_documents(minimal_docs)
    return texts_chunk

In [18]:
texts_chunk = text_split(filter_data)
print(f"Number of chunks: {len(texts_chunk)}")

Number of chunks: 81130


In [19]:
texts_chunk

[Document(metadata={'source': 'data\\B D Chaurasia’s Human Anatomy Regional & Applied Dissection.pdf', 'page': 1}, page_content='Regional and Applied Dissection and Clinical\nV olume3\nHuman\nAnatomy\nBD Chaurasia’s Eighth\nEdition\nHead and Neck\nAs per Medical Council of India: Competency based Undergraduate Curriculum for the Indian Medical Graduate , 2018'),
 Document(metadata={'source': 'data\\B D Chaurasia’s Human Anatomy Regional & Applied Dissection.pdf', 'page': 2}, page_content='Dr BD Chaurasia (1937–1985)\nwas Reader in Anatomy at GR Medical College, Gwalior.\nHe received his MBBS in 1960, MS in 1965 and PhD in 1975.\nHe was elected fellow of National Academy of Medical Sciences (India) in 1982.\nHe was a member of the Advisory Board of the Acta Anatomica since 1981,\nmember of the editorial board of Bionature, and in addition\nmember of a number of scientific societies.\nHe had a large number of research papers to his credit.'),
 Document(metadata={'source': 'data\\B D Chau

In [21]:
from langchain_huggingface import HuggingFaceEmbeddings

def download_embeddings():
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
    embeddings = HuggingFaceEmbeddings(
        model_name = model_name
    )
    return embeddings

embeddings = download_embeddings()

In [22]:
embeddings

HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

In [43]:
vector = embeddings.embed_query("Hello World")
vector

[-0.03447720408439636,
 0.031023239716887474,
 0.00673496862873435,
 0.026108969002962112,
 -0.03936196118593216,
 -0.16030246019363403,
 0.06692393124103546,
 -0.0064414795488119125,
 -0.047450557351112366,
 0.014758911915123463,
 0.0708753690123558,
 0.05552756413817406,
 0.01919337548315525,
 -0.026251327246427536,
 -0.010109500028192997,
 -0.026940541341900826,
 0.022307470440864563,
 -0.02222665585577488,
 -0.14969269931316376,
 -0.01749308407306671,
 0.007676247972995043,
 0.054352279752492905,
 0.003254473675042391,
 0.03172597661614418,
 -0.0846213549375534,
 -0.0294059906154871,
 0.051595624536275864,
 0.048124030232429504,
 -0.003314792178571224,
 -0.05827920511364937,
 0.04196930304169655,
 0.022210685536265373,
 0.1281888484954834,
 -0.02233896590769291,
 -0.011656301096081734,
 0.06292833387851715,
 -0.032876282930374146,
 -0.09122605621814728,
 -0.031175389885902405,
 0.05269956216216087,
 0.0470348559319973,
 -0.08420302718877792,
 -0.03005620837211609,
 -0.0207447819411

In [45]:
print("vector length", len(vector))

vector length 384


In [23]:
from dotenv import load_dotenv
import os
# Load variables from a .env file
load_dotenv()

True

In [24]:
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
GROQ_API_KEY= os.getenv("GROQ_API_KEY")

os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY
os.environ["GROQ_API_KEY"] = GROQ_API_KEY

In [25]:
from pinecone import Pinecone
pinecone_api_key = PINECONE_API_KEY

pc = Pinecone(api_key=pinecone_api_key)

In [26]:
pc

In [27]:
from pinecone import ServerlessSpec
index_name = "vitaai"

if not pc.has_index(index_name):
    pc.create_index(
        name = index_name,
        dimension=384,   #Dimension of the embeddings
        metric="cosine",   #cosine similarity
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )

index = pc.Index(index_name)


In [28]:
# embedd each chunk and upsert the embeddings into your pinecone index.
from langchain_pinecone import PineconeVectorStore
docsearch = PineconeVectorStore.from_documents(
    documents=texts_chunk,
    embedding=embeddings, 
    index_name=index_name
)

In [29]:
# Load existing index

from langchain_pinecone import PineconeVectorStore
docsearch = PineconeVectorStore.from_existing_index(
    index_name=index_name,
    embedding=embeddings
)


Add more data to the existing Pinecone index

In [30]:
about = Document(
    page_content="""
    Identity: My name is VitaAI. I am an advanced virtual health assistant designed to provide medical information, symptom analysis, and wellness guidance. I am not a human doctor, but an AI trained to assist with health-related queries.
    
    Creator: VitaAI was created and developed by an engineering team led by Harsh, with key contributions from Abhishek and Sonu. The project was supervised by Dr. Abhaya. The goal was to build a responsive, accessible, and accurate tool for preliminary health assessment.
    
    Capabilities: I can assist users by analyzing symptoms, suggesting potential causes for common ailments, explaining medical terminology, and providing general advice on diet, nutrition, and mental wellness. I can also help interpret lab report metrics.
    
    Safety Disclaimer: It is important to know that I am an Artificial Intelligence, not a licensed medical professional. My responses are for informational purposes only and should never replace professional medical advice, diagnosis, or treatment.
    
    Technology: Under the hood, VitaAI utilizes Large Language Models (LLMs) and Vector Search technology to retrieve accurate medical context.
    
    Privacy: VitaAI is designed with privacy in mind. I do not store personal identifiable information (PII) permanently for training purposes.
    """,
    metadata={"source": "About"}
)

In [31]:
docsearch.add_documents(documents=[about])

['6f71a27e-3c39-4c8a-9e95-896c3fe30f5d']

In [32]:
retriever = docsearch.as_retriever(search_type="similarity", search_kwargs={"k":3})

In [33]:
retrieved_docs = retriever.invoke("What is Acne?")
retrieved_docs

[Document(id='def2ac83-bd20-473f-b2f9-021f6bd53f09', metadata={'page': 39.0, 'source': 'data\\The Gale Encyclopedia of Medicine.pdf'}, page_content='GALE ENCYCLOPEDIA OF MEDICINE 226\nAcne\nGEM - 0001 to 0432 - A  10/22/03 1:41 PM  Page 26'),
 Document(id='30cb3ddf-e2b6-431e-96d6-1c4a3f9f7ea0', metadata={'page': 38.0, 'source': 'data\\The Gale Encyclopedia of Medicine.pdf'}, page_content='GALE ENCYCLOPEDIA OF MEDICINE 2 25\nAcne\nAcne vulgaris affecting a woman’s face. Acne is the general\nname given to a skin disorder in which the sebaceous\nglands become inflamed. (Photograph by Biophoto Associ-\nates, Photo Researchers, Inc. Reproduced by permission.)\nGEM - 0001 to 0432 - A  10/22/03 1:41 PM  Page 25'),
 Document(id='20bb7735-91b2-49cf-8692-d34f313d0887', metadata={'page': 37.0, 'source': 'data\\The Gale Encyclopedia of Medicine.pdf'}, page_content='Acidosis see Respiratory acidosis; Renal\ntubular acidosis; Metabolic acidosis\nAcne\nDefinition\nAcne is a common skin disease charac

In [58]:
from langchain_groq import ChatGroq
chatModel = ChatGroq(model="llama-3.3-70b-versatile",temperature=0.2)


In [59]:

from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

from langchain_core.prompts import ChatPromptTemplate

In [75]:
system_prompt = """You are a clinical medical assistant.

Answer the question strictly using the provided context.
If the information is insufficient, respond with:
"I don't know based on the retrieved context."

Rules:
1. Maximum 3 sentences.
2. No assumptions.
3. No external knowledge.
4. Always include source and page at the end.

Format your response exactly like this:

<Answer>

Sources:
- <source>, Page <page>

Context:
{context}
"""
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}")

    ]
)

In [76]:
question_answer_chain = create_stuff_documents_chain(chatModel, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

In [77]:

response = rag_chain.invoke({"input": "what is thymus"})


print(response["answer"])

The thymus is an important lymphoid organ situated in the anterior and superior mediastinum of the thorax. It is a bilobed structure made up of two pyramidal lobes of unequal size connected together by areolar tissue. The thymus is well developed at birth, continues to grow up to puberty, and thereafter undergoes gradual atrophy and replacement by fat.

Sources:
- Context, Page 1
